# 01 - Data Download and Cleaning

This notebook downloads the UCI Appliances Energy Prediction dataset, inspects it for missing values and gaps, and resamples it from 10-minute to hourly resolution.

Reusable logic lives in `src/appliance_energy/data.py`; this notebook calls those functions and narrates the results.

In [1]:
import sys
from pathlib import Path

# Make src/ importable from within notebooks/
sys.path.insert(0, str(Path.cwd().parent / "src"))

import warnings
warnings.filterwarnings("ignore")

from appliance_energy.data import (
    download_raw_data,
    load_raw_data,
    report_missing_and_gaps,
    resample_to_hourly,
)

## Download and load the raw 10-minute data

In [2]:
download_raw_data()
raw = load_raw_data()

print("Raw data shape:", raw.shape)
raw.head()

Using cached raw data: C:\Users\Hp\Downloads\appliance-energy-forecasting\project\data\raw\energydata_complete.csv
Raw data shape: (19735, 28)


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,45.566667,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,45.992500,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,45.890000,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,45.723333,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,45.530000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


## Check for missing values and gaps in the sampling grid

The dataset should be sampled every 10 minutes with no gaps. Any missing timestamps would need to be handled before resampling.

In [3]:
missing_stamps = report_missing_and_gaps(raw)


Missing values per column (non-zero only):
  none

Expected 10min timestamps: 19735
Actual rows:               19735
Missing timestamps:        0


## Resample to hourly means

Hourly resolution keeps the daily usage cycle visible while making SARIMAX with a 24-period seasonal component computationally tractable (144 ten-minute steps per day would make a seasonal ARIMA very slow to fit). Any small resulting gaps are filled with time-based interpolation.

In [4]:
hourly = resample_to_hourly(raw)
hourly.head()


Hourly data shape: (3290, 28)
Period: 2016-01-11 17:00:00 to 2016-05-27 18:00:00


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,55.000000,35.000000,19.890000,46.502778,19.200000,44.626528,19.790000,44.897778,18.932778,45.738750,...,17.016667,45.446667,6.308333,733.750000,92.000000,6.166667,53.416667,5.050000,26.823044,26.823044
2016-01-11 18:00:00,176.666667,51.666667,19.897778,45.879028,19.268889,44.438889,19.770000,44.863333,18.908333,46.066667,...,16.981667,45.290000,5.941667,734.266667,91.583333,5.416667,40.000000,4.658333,22.324206,22.324206
2016-01-11 19:00:00,173.333333,25.000000,20.495556,52.805556,19.925556,46.061667,20.052222,47.227361,18.969444,47.815556,...,16.902222,45.311389,6.000000,734.791667,89.750000,6.000000,40.000000,4.391667,33.734932,33.734932
2016-01-11 20:00:00,125.000000,35.000000,20.961111,48.453333,20.251111,45.632639,20.213889,47.268889,19.190833,49.227917,...,16.890000,45.118889,6.000000,735.283333,87.583333,6.000000,40.000000,4.016667,25.679642,25.679642
2016-01-11 21:00:00,103.333333,23.333333,21.311667,45.768333,20.587778,44.961111,20.373333,46.164444,19.425556,47.918889,...,16.890000,44.807778,5.833333,735.566667,87.416667,6.000000,40.000000,3.816667,18.826274,18.826274


## Summary

- Raw data: 10-minute resolution, ~4.5 months of observations.
- No missing values or gaps in the sampling grid were found in the raw data.
- Data resampled to hourly means and saved to `data/processed/appliance_hourly.csv` for use in subsequent notebooks and scripts.